(wikipedia)=
# Uso de la wikipedia con LangChain

En esta sección construiremos un **agente inteligente** capaz de consultar
Wikipedia en tiempo real para responder preguntas con información veraz y actualizada.


## ¿Por qué conectar un LLM a Wikipedia?
```{index} wikipedia
```
Los modelos de lenguaje como Groq son extraordinariamente capaces, pero tienen una limitación fundamental: su conocimiento está **congelado en el tiempo**.

Todo lo que saben proviene de los datos con los que fueron entrenados, y a partir de esa fecha de corte, el mundo sigue cambiando pero el modelo no.

Wikipedia nos ofrece algo que el LLM por sí solo no puede garantizar:

-  **Información verificable** — cada artículo está respaldado por fuentes citadas

-  **Cobertura enciclopédica** — más de 60 millones de artículos en 300 idiomas

-  **Contenido actualizado** — la comunidad edita y corrige artículos continuamente

-  **Acceso completamente libre** — sin coste, sin restricciones, con API pública

Al combinar Wikipedia con un LLM conseguimos lo mejor de ambos mundos: la **capacidad de razonamiento y expresión** del modelo junto con la **fiabilidad y amplitud** de una enciclopedia colaborativa global.


## ¿Qué vamos a construir?

A lo largo de esta lección crearemos paso a paso un **agente conversacional** que:

1. Recibe una pregunta en lenguaje natural 

2. Decide de forma autónoma si necesita consultar Wikipedia para responderla

3. Lanza la búsqueda en Wikipedia con el término más adecuado

4. Lee el resultado y lo utiliza como base para elaborar una respuesta clara y precisa

5. Devuelve la respuesta al usuario, fundamentada en información real

Todo esto con tres ingredientes: la librería **LangChain**, el LLM ultrarrápido **Groq** y la herramienta **WikipediaQueryRun**, que actúa como puente directo a la enciclopedia más grande del mundo.


Para conseguir todo esto las dependencias que se necesitan son las siguientes:

```
!pip install langchain langchain-community langchain-groq wikipedia wikipedia-api requests
```

In [1]:
# ==============================================================
# Configuración de la API Key de Groq
# ==============================================================
# Obtén tu API key gratuita en: https://console.groq.com/keys


import os
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"]= GROQ_API_KEY

from langchain_groq import ChatGroq
llm=ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0.2
    )

# Verificación rápida
respuesta = llm.invoke("Di solo 'Groq conectado correctamente' y nada más.")
print(respuesta.content)   

Groq conectado correctamente



## La herramienta WikipediaQueryRun
```{index} WikipediaQueryRun
```

LangChain incluye `WikipediaQueryRun`, una **Tool** lista para usar que:

1. Recibe un término de búsqueda

2. Consulta la API oficial de Wikipedia

3. Devuelve un resumen del artículo más relevante

**Parámetros configurables de WikipediaAPIWrapper:**

| Parámetro | Descripción | Valor por defecto |
|---|---|---|
| `lang` | Idioma de Wikipedia | `"en"` |
| `top_k_results` | Cuántos artículos considerar | `3` |
| `doc_content_chars_max` | Máximo de caracteres a devolver | `4000` |


**A TENER EN CUENTA**:    
          
> Esta herramienta NO carga el artículo completo, sino un resumen  controlado por `doc_content_chars_max`. Es suficiente para que  el LLM tenga contexto sin saturar la ventana de tokens.


In [5]:
# ==============================================================
# Configuración de la herramienta Wikipedia
# ==============================================================

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Configuramos el wrapper de Wikipedia
wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        lang="es",                  # Wikipedia en español
        top_k_results=2,            # Considera los 2 artículos más relevantes
        doc_content_chars_max=3000  # Máximo 3000 caracteres por consulta
    )
)

# Probamos la herramienta directamente (sin LLM todavía)
print("Prueba directa de la herramienta Wikipedia:")
print("-" * 50)
resultado = wikipedia.run("Redes neuronales artificiales")
print(resultado[:800], "...")  # Mostramos solo los primeros 800 caracteres

Prueba directa de la herramienta Wikipedia:
--------------------------------------------------
Page: Red neuronal artificial
Summary: En el aprendizaje automático, una red neuronal artificial (abreviada ANN o NN) es un modelo dentro de los llamados sistemas conexionistas, inspirado en la estructura y función de las redes neuronales biológicas en los cerebros animales. Una ANN consta de unidades o nodos conectados llamados neuronas artificiales, que modelan vagamente las neuronas del cerebro. Estas están conectadas por aristas, que modelan las sinapsis del cerebro. Cada neurona artificial recibe «señales» de las neuronas conectadas, luego las procesa y envía una señal a otras neuronas conectadas. La «señal» es un número real, y la salida de cada neurona se calcula mediante una función no lineal de la suma de sus entradas, llamada función de activación. La fuerza de la señal en cada c ...



## Construyendo el Agente

Un **agente** en LangChain es un LLM que puede decidir autónomamente qué herramientas usar y cuándo usarlas.



In [16]:
from langchain_core.tools import tool

@tool
def wikipedia(query: str) -> str:
    """
    Útil para buscar información factual sobre personas, lugares,
    conceptos científicos, eventos históricos y cualquier tema enciclopédico.
    El input debe ser el término de búsqueda en español, lo más conciso posible.
    """
    import requests
    url = f"https://es.wikipedia.org/api/rest_v1/page/summary/{query.replace(' ', '_')}"
    headers = {"User-Agent": "LangChainLesson/1.0 (educational use)"}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        titulo   = data.get("title", "Sin título")
        extracto = data.get("extract", "Sin contenido disponible.")
        url_wiki = data.get("content_urls", {}).get("desktop", {}).get("page", "")
        return f"{titulo}\n\n{extracto[:3000]}\n\nFuente: {url_wiki}"
    except requests.exceptions.HTTPError:
        return f"No se encontró el artículo '{query}' en Wikipedia."
    except Exception as e:
        return f"Error al consultar Wikipedia: {e}"

herramientas = [wikipedia]

In [22]:
# ==============================================================
# Construcción del agente ReAct con Wikipedia
# ==============================================================
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent   

# Creamos el agente con la API moderna de LangGraph
# No necesita prompt manual ni AgentExecutor
agente = create_agent(
    model=llm,                  # El ChatGroq que ya tenemos configurado
    tools=herramientas,         # La lista con wikipedia_tool
)

print(" Agente LangGraph listo.")

 Agente LangGraph listo.


In [23]:
resultado = agente.invoke({
    "messages": [HumanMessage(content="¿Qué es una red neuronal artificial?")]
})
print("\n Respuesta:")
print(resultado["messages"][-1].content)


 Respuesta:
Una red neuronal artificial es un modelo de aprendizaje automático inspirado en la estructura y función de las redes neuronales biológicas en los cerebros animales. Está compuesta por unidades o nodos conectados llamados neuronas artificiales, que modelan vagamente las neuronas del cerebro. Estas neuronas están conectadas por aristas, que modelan las sinapsis del cerebro, y cada neurona recibe señales de las neuronas conectadas, las procesa y envía señales a otras neuronas conectadas. La fuerza de la señal en cada conexión está determinada por un peso, que se ajusta durante el proceso de aprendizaje.
